# A Priori Neural Network Correction for 1D Viscous Burgers Equation

This notebook extends the corrective-term idea from linear advection to a **nonlinear** PDE.
The Burgers equation develops steep gradients (near-shocks) that make correction harder —
the NN must learn context-dependent corrections that vary across the solution.

**PDE:** $\quad u_t + u\,u_x = \nu\,u_{xx}$ on a periodic domain, $\nu = 0.02$.

**Setup:**
- **Fine grid:** $N_f = 512$ points, spectral differentiation + RK4 (resolves the shock region)
- **Coarse grid:** $N_c = 64$ points, upwind FD for convection + central FD for diffusion + RK4
- **NN correction:** learns $\delta_j = \bar{u}_j^{\text{fine}} - u_j^{\text{coarse}}$ from a 5-point stencil + derivatives

Key difference from the advection case: the error is no longer a simple diffusive smearing.
Near the shock front, the coarse solver both smears *and* mislocates the steep gradient.
The network needs richer features (wider stencil) to handle this.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from scipy.fft import fft, ifft, fftfreq

%matplotlib inline
plt.rcParams['figure.dpi'] = 130

torch.manual_seed(0)
np.random.seed(0)

## 1. Parameters

The viscosity $\nu = 0.02$ is small enough that the sine initial condition steepens
into a near-shock by $T = 1.5$.  The fine solver uses a spectral method (exact
derivatives in Fourier space) while the coarse solver uses finite differences —
this mismatch is intentional and representative of real applications where the
coarse solver uses a cheaper discretisation.

In [ ]:
L     = 2 * np.pi    # domain length (periodic)
nu    = 0.02         # viscosity (small enough to get near-shock)
N_f   = 512
N_c   = 64
T     = 1.5          # end time (shock forms around t~1.0 for sin IC)
dt_f  = 1e-4
dt_c  = 5e-4
N_IC  = 300
N_test = 40
EPOCHS = 400

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
ratio  = N_f // N_c

x_f = np.linspace(0, L, N_f, endpoint=False)
x_c = np.linspace(0, L, N_c, endpoint=False)
dx_c = L / N_c

print(f"Using device: {DEVICE}")
print(f"Fine grid: {N_f} points, dt={dt_f}")
print(f"Coarse grid: {N_c} points, dt={dt_c}")
print(f"Refinement ratio: {ratio}x")

## 2. Numerical Schemes

### Fine solver: spectral + RK4

The spectral method computes derivatives exactly in Fourier space:
$$\widehat{u_x} = i k \, \hat{u}, \qquad \widehat{u_{xx}} = -k^2 \, \hat{u}$$

This gives the RHS of Burgers' equation: $-u\,u_x + \nu\,u_{xx}$.

### Coarse solver: finite differences + RK4

The coarse solver uses **upwind differencing** for the nonlinear convection term
(switching direction based on the sign of $u$) and central differences for diffusion.
The upwind scheme adds artificial diffusion, and the coarser grid cannot resolve the
steep gradient region — both contribute to the error the NN will learn to correct.

In [ ]:
def burgers_rhs_spectral(u, nu, N):
    """Spectral RHS for Burgers: -u*du/dx + nu*d2u/dx2."""
    u_hat   = fft(u)
    k       = fftfreq(N, d=1.0/N)
    du_hat  = 1j * k * u_hat
    d2u_hat = -(k**2) * u_hat
    du_dx   = np.real(ifft(du_hat))
    d2u_dx2 = np.real(ifft(d2u_hat))
    return -u * du_dx + nu * d2u_dx2

def rk4_step(u, rhs_fn, dt, **kwargs):
    k1 = rhs_fn(u,         **kwargs)
    k2 = rhs_fn(u+dt/2*k1, **kwargs)
    k3 = rhs_fn(u+dt/2*k2, **kwargs)
    k4 = rhs_fn(u+dt*k3,   **kwargs)
    return u + dt/6*(k1 + 2*k2 + 2*k3 + k4)

def simulate_fine(u0):
    u = u0.copy()
    t = 0.0
    while t < T - 1e-10:
        dt = min(dt_f, T - t)
        u  = rk4_step(u, burgers_rhs_spectral, dt, nu=nu, N=N_f)
        t += dt
    return u

def burgers_rhs_fd(u, nu, dx):
    """FD RHS: upwind for convection, central for diffusion."""
    conv = np.where(u > 0,
                    u * (u - np.roll(u,  1)) / dx,
                    u * (np.roll(u, -1) - u) / dx)
    diff = nu * (np.roll(u,-1) - 2*u + np.roll(u,1)) / dx**2
    return -conv + diff

def simulate_coarse(u0):
    u = u0.copy()
    t = 0.0
    while t < T - 1e-10:
        dt = min(dt_c, T - t)
        u  = rk4_step(u, burgers_rhs_fd, dt, nu=nu, dx=dx_c)
        t += dt
    return u

def coarsen(u_fine, ratio):
    return u_fine.reshape(-1, ratio).mean(axis=1)

## 3. The Problem: Shock Steepening and Numerical Smearing

Before training, let's see what happens with the classic $u_0 = \sin(x)$ initial condition.
The nonlinear term $u\,u_x$ causes the wave to steepen — the crest moves faster than the
trough.  By $T = 1.5$, a thin viscous shock layer has formed.

The fine solver captures this; the coarse solver smears it.

In [ ]:
u0_demo_f = np.sin(x_f)
u0_demo_c = coarsen(u0_demo_f, ratio)

u_demo_fine   = simulate_fine(u0_demo_f)
u_demo_coarse = simulate_coarse(u0_demo_c)
u_demo_fine_c = coarsen(u_demo_fine, ratio)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(x_f, u0_demo_f, 'g--', lw=1, alpha=0.5, label='IC: sin(x)')
ax.plot(x_f, u_demo_fine, 'k-', lw=2, label='Fine (truth)')
ax.plot(x_c, u_demo_coarse, 'r--', lw=1.8, label='Coarse (upwind FD)')
ax.set_title(f'Burgers at T={T} — shock steepening')
ax.set_xlabel('x'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(x_c, u_demo_fine_c - u_demo_coarse, 'r-', lw=1.5)
ax.axhline(0, color='k', lw=0.8)
ax.set_title('Pointwise error (fine − coarse)')
ax.set_xlabel('x'); ax.set_ylabel('Error'); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"RMSE of coarse vs fine: {np.mean((u_demo_fine_c - u_demo_coarse)**2)**0.5:.4e}")
print(f"\nNotice how the error is concentrated near the shock — the correction")
print(f"needs to be spatially varying, not a uniform shift.")

## 4. Feature Extraction

For the advection problem, 3 features sufficed ($u$, $du/dx$, $d^2u/dx^2$).
For Burgers, we use a richer feature set:

| Feature | Description | Why? |
|---------|-------------|------|
| $u_{j-2}, u_{j-1}, u_j, u_{j+1}, u_{j+2}$ | 5-point stencil | Captures local solution shape and asymmetry near shocks |
| $du/dx$ | Central difference | Gradient magnitude — large at shocks |
| $d^2u/dx^2$ | Central second difference | Curvature — changes sign at inflection points |

Total: **7 inputs**.  The wider stencil lets the network "see" the shock approaching
from either side, not just the local curvature.

In [ ]:
def extract_features(u, dx):
    """
    Local stencil of 5 cells + first and second derivatives.
    Returns array of shape (N, 7).
    """
    um2 = np.roll(u, 2)
    um1 = np.roll(u, 1)
    up1 = np.roll(u, -1)
    up2 = np.roll(u, -2)
    du_dx   = (up1 - um1) / (2*dx)
    d2u_dx2 = (up1 - 2*u + um1) / dx**2
    return np.stack([um2, um1, u, up1, up2, du_dx, d2u_dx2], axis=1)

## 5. Training Data: Random Initial Conditions

We generate 340 random ICs (300 train, 40 test) from three families:
- **Single Gaussian pulses** — smooth, localised
- **Sinusoidal** $u_0 = A\sin(x)$ — the classic steepening case
- **Double Gaussians** — two bumps that interact

This variety is important: the network should learn the *scheme's error structure*,
not memorise corrections for a single IC shape.

In [ ]:
rng = np.random.default_rng(1)

def random_ic(x, rng):
    """Mix of single Gaussian pulses and smooth sinusoidal ICs."""
    choice = rng.integers(0, 3)
    if choice == 0:
        mu    = rng.uniform(np.pi*0.5, np.pi*1.5)
        sigma = rng.uniform(0.3, 0.8)
        return np.exp(-((x - mu)**2) / (2*sigma**2))
    elif choice == 1:
        amp = rng.uniform(0.5, 1.5)
        return amp * np.sin(x)
    else:
        u = np.zeros_like(x)
        for _ in range(2):
            mu    = rng.uniform(0.5, 5.5)
            sigma = rng.uniform(0.2, 0.6)
            amp   = rng.uniform(0.3, 1.0)
            u    += amp * np.exp(-((x - mu)**2) / (2*sigma**2))
        return u

In [ ]:
print("Generating dataset (may take ~1-2 min) ...")

X_list, Y_list = [], []

for i in range(N_IC + N_test):
    u0_f  = random_ic(x_f, rng)
    u0_c  = coarsen(u0_f, ratio)

    u_f   = simulate_fine(u0_f)
    u_c   = simulate_coarse(u0_c)

    u_f_coarse = coarsen(u_f, ratio)

    feat   = extract_features(u_c, dx_c)
    target = u_f_coarse - u_c

    X_list.append(feat)
    Y_list.append(target)

    if (i+1) % 50 == 0:
        print(f"  Generated {i+1}/{N_IC+N_test}")

X_all = np.array(X_list, dtype=np.float32)
Y_all = np.array(Y_list, dtype=np.float32)

X_train = torch.tensor(X_all[:N_IC].reshape(-1, 7)).to(DEVICE)
Y_train = torch.tensor(Y_all[:N_IC].reshape(-1)).unsqueeze(1).to(DEVICE)
X_test  = torch.tensor(X_all[N_IC:].reshape(-1, 7)).to(DEVICE)
Y_test  = torch.tensor(Y_all[N_IC:].reshape(-1)).unsqueeze(1).to(DEVICE)

print(f"\nTraining samples: {X_train.shape[0]}  (= {N_IC} ICs × {N_c} points)")
print(f"Test samples:     {X_test.shape[0]}")

## 6. The Correction Network

A wider MLP than the advection case: 7 inputs → 128 → 128 → 64 → 1.

The larger hidden layers (128 vs 64) reflect the harder mapping — near a shock,
the relationship between the stencil values and the required correction is more
complex than the smooth diffusive correction in the advection case.

We also add a small weight decay ($10^{-5}$) to regularise, since the training
set is larger and we want to avoid overfitting to specific IC realisations.

In [ ]:
class BurgersCorrectionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(7, 128),  nn.Tanh(),
            nn.Linear(128, 128), nn.Tanh(),
            nn.Linear(128, 64),  nn.Tanh(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)

model     = BurgersCorrectionNet().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, EPOCHS)
loss_fn   = nn.MSELoss()

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}")
print(model)

## 7. Training

In [ ]:
print("Training ...")
train_losses, test_losses = [], []

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad()
    loss = loss_fn(model(X_train), Y_train)
    loss.backward()
    optimizer.step()
    scheduler.step()

    model.eval()
    with torch.no_grad():
        test_loss = loss_fn(model(X_test), Y_test).item()
    train_losses.append(loss.item())
    test_losses.append(test_loss)

    if (epoch+1) % 100 == 0:
        print(f"  Epoch {epoch+1:4d}  train={loss.item():.2e}  test={test_loss:.2e}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.semilogy(train_losses, label='Train', alpha=0.7)
ax.semilogy(test_losses,  label='Test',  alpha=0.7)
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss')
ax.set_title('Training convergence')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Evaluation: Classic Steepening IC

We test on the canonical $u_0 = \sin(x)$ initial condition.  By $T = 1.5$, the
nonlinear steepening has produced a thin shock layer where $u$ drops sharply.
The coarse solver smears this transition; the NN should sharpen it back up.

In [ ]:
model.eval()

u0_f  = np.sin(x_f)
u0_c  = coarsen(u0_f, ratio)

u_f   = simulate_fine(u0_f)
u_c   = simulate_coarse(u0_c)
u_f_c = coarsen(u_f, ratio)

feat  = extract_features(u_c, dx_c).astype(np.float32)
with torch.no_grad():
    delta = model(torch.tensor(feat).to(DEVICE)).cpu().numpy().flatten()

u_corr = u_c + delta

## 9. Results

**Left:** solutions.  **Centre:** pointwise error.  **Right:** RMSE bar chart.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

ax = axes[0]
ax.plot(x_f, simulate_fine(np.sin(x_f)), 'k-', lw=2, label='Fine (truth)')
ax.plot(x_c, u_c,    'r--', lw=1.8, label='Coarse FD')
ax.plot(x_c, u_corr, 'b-',  lw=1.8, label='Coarse + NN')
ax.set_title(f'Burgers at T={T:.1f} (sin IC)')
ax.set_xlabel('x'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(x_c, u_f_c - u_c,    'r--', label='Error (no correction)')
ax.plot(x_c, u_f_c - u_corr, 'b-',  label='Error (with NN)')
ax.axhline(0, color='k', lw=0.8)
ax.set_title('Pointwise error'); ax.set_xlabel('x')
ax.legend(); ax.grid(True, alpha=0.3)

rmse_before = np.mean((u_f_c - u_c)**2)**0.5
rmse_after  = np.mean((u_f_c - u_corr)**2)**0.5

ax = axes[2]
ax.bar(['Before', 'After'],
       [rmse_before, rmse_after],
       color=['red','blue'], alpha=0.7)
ax.set_ylabel('RMSE'); ax.set_title('Error reduction')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"RMSE before correction: {rmse_before:.4e}")
print(f"RMSE after  correction: {rmse_after:.4e}")
print(f"Improvement factor:     {rmse_before/rmse_after:.1f}x")

## 10. Stress Test: Double Gaussian

While the sinusoidal IC was in the training distribution, let's try a **double Gaussian**
with parameters not seen during training.  Two bumps steepen independently and may
interact — a harder test of generalisation.

In [ ]:
# Double Gaussian stress test
u0_stress_f = 0.8 * np.exp(-((x_f - 1.5)**2) / (2*0.4**2)) + \
              1.2 * np.exp(-((x_f - 4.5)**2) / (2*0.3**2))
u0_stress_c = coarsen(u0_stress_f, ratio)

u_stress_fine   = simulate_fine(u0_stress_f)
u_stress_coarse = simulate_coarse(u0_stress_c)
u_stress_fine_c = coarsen(u_stress_fine, ratio)

feat_stress = extract_features(u_stress_coarse, dx_c).astype(np.float32)
with torch.no_grad():
    delta_stress = model(torch.tensor(feat_stress).to(DEVICE)).cpu().numpy().flatten()

u_stress_corr = u_stress_coarse + delta_stress

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(x_c, u_stress_fine_c,  'k-',  lw=2,   label='Fine (truth)')
ax.plot(x_c, u_stress_coarse,  'r--', lw=1.8, label='Coarse FD')
ax.plot(x_c, u_stress_corr,    'b-',  lw=1.8, label='Coarse + NN')
ax.set_title(f'Double Gaussian at T={T}')
ax.set_xlabel('x'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(x_c, u_stress_fine_c - u_stress_coarse, 'r--', label='Error without NN')
ax.plot(x_c, u_stress_fine_c - u_stress_corr,   'b-',  label='Error with NN')
ax.axhline(0, color='k', lw=0.8)
ax.set_title('Pointwise error (double Gaussian)')
ax.set_xlabel('x'); ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

err_s_before = np.mean((u_stress_fine_c - u_stress_coarse)**2)**0.5
err_s_after  = np.mean((u_stress_fine_c - u_stress_corr)**2)**0.5
print(f"Double Gaussian RMSE before: {err_s_before:.4e}")
print(f"Double Gaussian RMSE after:  {err_s_after:.4e}")
if err_s_after < err_s_before:
    print(f"Improvement: {err_s_before/err_s_after:.1f}x")
else:
    print(f"Degradation: {err_s_after/err_s_before:.2f}x")

## 11. Network Introspection: Feature Sensitivity

We sweep each of the 7 input features while holding the others at zero to see
which features the network relies on most.  For Burgers, we expect the stencil
asymmetry (difference between left and right neighbours) and the first derivative
to be important, since these detect the shock location and direction.

In [ ]:
feature_names = ['$u_{j-2}$', '$u_{j-1}$', '$u_j$', '$u_{j+1}$', '$u_{j+2}$',
                 '$du/dx$', '$d^2u/dx^2$']

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i in range(7):
    sweep = np.linspace(-2, 2, 200)
    inp = np.zeros((200, 7), dtype=np.float32)
    inp[:, i] = sweep
    with torch.no_grad():
        out = model(torch.tensor(inp).to(DEVICE)).cpu().numpy().flatten()
    axes[i].plot(sweep, out, 'b-', lw=1.5)
    axes[i].set_xlabel(feature_names[i]); axes[i].set_ylabel('NN output')
    axes[i].set_title(f'Response to {feature_names[i]}')
    axes[i].grid(True, alpha=0.3)
    axes[i].axhline(0, color='k', lw=0.5)
    axes[i].axvline(0, color='k', lw=0.5)
    axes[i].set_ylim([-.5,.5])
    axes[i].set_xlim([-2,2])

axes[7].axis('off')  # unused subplot
plt.suptitle('Network response to each input feature (others = 0)', fontsize=12)
plt.tight_layout()
plt.show()